In [1]:
import logging
import sys


# Konfiguration für Jupyter erzwingen
logging.basicConfig(
    level=logging.INFO,  # Auf welchen Level wird geloggt 
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout, 
    force=True         
)

# Logger erstellen
logger = logging.getLogger(__name__)

# Testen
logger.debug("Das ist eine Debug-Nachricht (nur für Entwickler).")
logger.info("Das Programm läuft ganz normal weiter.")
logger.warning("Achtung, hier könnte bald ein Problem auftreten!")

16:27:04 - __main__ - INFO - Das Programm läuft ganz normal weiter.
16:27:04 - __main__ - WARNING - Achtung, hier könnte bald ein Problem auftreten!


In [2]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pickle
import gc
import pandas as pd

# ---------------------------------------------------------
# Schritt 1: Master-Tabelle aufbauen (z.B. aus Population)
# ---------------------------------------------------------
with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/germany_graph_with_population.pkl", 'rb') as f:
    temp_data = pickle.load(f)

# Wir nehmen diesen DataFrame als unsere Basis (hier sind u, v, arc_id schon drin)
df_master_edges = temp_data['edges'][['arc_id', 'u', 'v', 'length_m', 'population']].copy()
df_master_nodes = temp_data['nodes'][['lat', 'lon']].copy()

# WICHTIG: Die große Original-Datei sofort wieder aus dem RAM werfen!
del temp_data
gc.collect()

# ---------------------------------------------------------
# Schritt 2: NUR die Unfalldaten (Accidents) extrahieren
# ---------------------------------------------------------
with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/germany_graph_with_accidents.pkl", 'rb') as f:
    temp_data = pickle.load(f)

# Wir extrahieren NUR die arc_id und den weighted_score
df_accidents_slim = temp_data['edges'][['arc_id', 'weighted_score']]

# Wir mergen den Score in unsere Master-Tabelle (anhand der arc_id)
df_master_edges = pd.merge(df_master_edges, df_accidents_slim, on='arc_id', how='left')

# Original-Datei wieder löschen
del temp_data, df_accidents_slim
gc.collect()


# ---------------------------------------------------------
# Schritt 3: NUR die Naturdaten (Nature) extrahieren
# ---------------------------------------------------------
with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/germany_graph_with_nature.pkl", 'rb') as f:
    temp_data = pickle.load(f)

# Wir extrahieren NUR die arc_id und den nature_score
df_nature_slim = temp_data['edges'][['arc_id', 'nature_score']]

# An die Master-Tabelle anfügen
df_master_edges = pd.merge(df_master_edges, df_nature_slim, on='arc_id', how='left')

# Original-Datei löschen
del temp_data, df_nature_slim
gc.collect()

with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/processed/all_data.pkl", 'rb') as f:
    all_data = pickle.load(f)

# --- NEU: LADEINFRASTRUKTUR BEREINIGEN ---
df_charge = all_data['charging_infrastructure'].copy()

# Kommas in Punkte umwandeln, damit Python rechnen kann
df_charge['Breitengrad'] = df_charge['Breitengrad'].astype(str).str.replace(',', '.').astype(float)
df_charge['Längengrad'] = df_charge['Längengrad'].astype(str).str.replace(',', '.').astype(float)

# (Optional) Nur Ladestationen ab 150 kW für LKWs zulassen
# df_charge = df_charge[df_charge['Nennleistung Ladeeinrichtung [kW]'] >= 150.0]

# ---------------------------------------------------------
# Schritt 4: Tunnel- und Straßendaten speicherschonend laden
# ---------------------------------------------------------
csv_geo_path = r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/edges_germnay_geo_optimized.csv"

print("1. Bereite arc_ids für den Filter vor...")
# Wir definieren als Menge (Set), welche Kanten überhaupt noch relevant sind.
# Sets sind für blitzschnelle Suchabfragen in Python optimiert.
valid_arc_ids = set(df_master_edges['arc_id'])

print("2. Lade Geo-Daten in Chunks und filtere sofort (RAM-Schutz)...")
chunksize = 2_000_000 # 2 Millionen Zeilen auf einmal laden
filtered_chunks = []

# Datentypen festlegen spart zusätzlich massiv RAM
# WICHTIG: Wenn deine arc_id eigentlich ein String ist, ändere 'int64' auf 'str'!
geo_dtypes = {
    'arc_id': 'int64', 
    'tunnel': 'str',
    'highway': 'str'
}

# Pandas iteriert durch die 5GB Datei, ohne sie komplett in den RAM zu laden
for chunk in pd.read_csv(csv_geo_path, usecols=['arc_id', 'tunnel', 'highway'], dtype=geo_dtypes, chunksize=chunksize):
    # SOFORT filtern: Behalte nur Zeilen, deren arc_id in unserem Master-Graph existiert
    chunk_filtered = chunk[chunk['arc_id'].isin(valid_arc_ids)]
    filtered_chunks.append(chunk_filtered)

print("3. Füge die gefilterten Daten zusammen...")
df_geo_all = pd.concat(filtered_chunks, ignore_index=True)

# Temporären Müll sofort löschen
del filtered_chunks
gc.collect()

print("4. Optimiere Datentypen für den Merge...")
# Verwandelt Strings in Kategorien (extrem speicherplatzsparend bei Werten wie 'yes', 'no', 'motorway')
df_geo_all['tunnel'] = df_geo_all['tunnel'].astype('category')
df_geo_all['highway'] = df_geo_all['highway'].astype('category')

print("5. Verarbeite finale Zusammenführung...")
df_master_edges = pd.merge(df_master_edges, df_geo_all, on='arc_id', how='left')

# Speicher sofort physisch freigeben
del df_geo_all
valid_arc_ids.clear()
gc.collect()

print("✅ Geo-Daten erfolgreich und ohne Absturz zusammengeführt!")




/tmp/ipykernel_80241/677402547.py:23: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  temp_data = pickle.load(f)


1. Bereite arc_ids für den Filter vor...
2. Lade Geo-Daten in Chunks und filtere sofort (RAM-Schutz)...
3. Füge die gefilterten Daten zusammen...
4. Optimiere Datentypen für den Merge...
5. Verarbeite finale Zusammenführung...
✅ Geo-Daten erfolgreich und ohne Absturz zusammengeführt!


In [4]:
import pandas as pd

print("="*90)
print("1. MASTER-GRAPH (KANTEN / EDGES)")
print("="*90)

# Welche Spalten haben wir jetzt alle erfolgreich zusammengeführt?
print(f"Spalten im Master-Graph:\n{df_master_edges.columns.tolist()}\n")

# Wie viele Kanten gibt es insgesamt?
print(f"Anzahl der Kanten im Netzwerk: {len(df_master_edges)}")

# Ein Blick auf die ersten 5 Zeilen (Alle wichtigen Metriken auf einen Blick!)
print(f"\nErste 5 Kanten (Head):\n{df_master_edges.head()}")

print("-" * 90)
print("STATISTIKEN FÜR DIE ZIELFUNKTION (Risk & Costs)")
print("-" * 90)

# Check der maximalen Scores (Wichtig für die spätere Normalisierung!)
print(f"Maximaler Unfall-Score (weighted_score): {df_master_edges['weighted_score'].max()}")
print(f"Maximaler Natur-Score (nature_score):    {df_master_edges['nature_score'].max()}")
print(f"Maximale Bevölkerung (population):       {df_master_edges['population'].max()}")
print(f"Längste Kante (length_m):              {df_master_edges['length_m'].max()} Meter")
# WICHTIG FÜR DEN SOLVER: Gibt es fehlende Werte (NaN)? 
# PuLP stürzt ab, wenn es mit NaN-Werten rechnen muss!
print("\nFehlende Werte (NaN) in den kritischen Spalten:")
print(df_master_edges[['weighted_score', 'nature_score', 'population', 'length_m']].isnull().sum())

# Falls es fehlende Werte gibt, füllen wir sie am besten mit 0 (oder einem anderen Standardwert)
# df_master_edges.fillna(0, inplace=True) 

print("="*90)
print("2. KNOTEN (NODES) & GEODATEN (aus berlin_graph_geo_com)")
print("="*90)
# Je nachdem, ob du die Knoten als DataFrame oder Dictionary hast:
# Falls Knoten im Master-Graph sind:
# print(f"Erste 5 Knoten:\n{df_master_nodes.head()}") 

# Tunnel-Status prüfen (Wichtig für ADR-Sperren)
# Da tunnel ein Dictionary war, schauen wir uns die Werte an:
if 'tunnel' in df_master_edges.columns:
    print(f"\nVerteilung der Tunnel im Netzwerk:\n{df_master_edges['tunnel'].value_counts()}")
else:
    print("\nHinweis: Tunnel-Daten sind noch nicht im df_master_edges (müssen ggf. noch gemergt werden).")

print("="*90)
print("3. KUNDEN, FAHRZEUGE & LIEFERUNGEN (ALL_DATA)")
print("="*90)

print(f"Verfügbare Datensätze in all_data: {list(all_data.keys())}\n")

print("Kundenstandorte (Erste 3):")
print(all_data['customer_locations'].head(3))
print(f"\nBeispiel-Ort (Index 1): {all_data['customer_locations'].iloc[1]['Ort']}\n")

print("-" * 40)
print("Fahrzeuge (Erste 3):")
print(all_data['vehicles'].head(3))

print("-" * 40)
print("Lieferungen (Erste 3):")
print(all_data['delivery_routes'].head(3))

print("-" * 40)
print("Ladestationen (Erste 3):")
print(all_data['charging_infrastructure'].head(3))

1. MASTER-GRAPH (KANTEN / EDGES)
Spalten im Master-Graph:
['arc_id', 'u', 'v', 'length_m', 'population', 'weighted_score', 'nature_score', 'highway', 'tunnel']

Anzahl der Kanten im Netzwerk: 3025675

Erste 5 Kanten (Head):
   arc_id           u           v    length_m  population  weighted_score  \
0       0  3786023396  3786023162  111.365417         0.0             0.0   
1       1    10210552   277783046   41.566623        29.0             0.0   
2       2   277783046    10210552   41.566623        29.0             0.0   
3       3   277783046  5186817684   24.481156        29.0             0.0   
4       4  5186817684   277783046   24.481156        29.0             0.0   

   nature_score      highway tunnel  
0      0.000655  residential     no  
1      0.001184  residential     no  
2      0.001184  residential     no  
3      0.001218  residential     no  
4      0.001218  residential     no  
-------------------------------------------------------------------------------------

In [5]:
# ================================
#  ORIGINALDATEN (DEUTSCHLAND)
# ================================

# Depot-Koordinaten (Kassel) festlegen
depot_lat = 51.3127
depot_lon = 9.4797

# Wir nutzen jetzt direkt das originale all_data Dictionary!
# (Stelle sicher, dass die CSV/Excel-Dateien für all_data in einer 
# Zelle weiter oben geladen wurden)

df_customers = all_data['customer_locations']
delivery_routes = all_data['delivery_routes']
vehicles = all_data['vehicles']

print(f"Es wurden {len(df_customers)} Kundenstandorte geladen.")
print(f"Es wurden {len(delivery_routes)} Lieferungen geladen.")

Es wurden 10 Kundenstandorte geladen.
Es wurden 10 Lieferungen geladen.


In [6]:
import pandas as pd
import numpy as np

# ================================
# DEPOT-KOORDINATEN (DEUTSCHLAND)
# ================================
depot_lat = 51.3127  
depot_lon = 9.4797   

# Vektorisierte Haversine-Funktion (rasend schnell!)
def vectorized_haversine(lat1, lon1, lat2_series, lon2_series):
    R = 6371000  # Erdradius in Metern
    
    # Alles in Bogenmaß umwandeln
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2_series), np.radians(lon2_series)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    return R * c

# Knotendaten
df_nodes = df_master_nodes

# Damit Numpy nicht meckert, stellen wir sicher, dass die Koordinaten floats sind
lat_series = df_nodes["lat"].astype(float)
lon_series = df_nodes["lon"].astype(float)

# ================================
# 1. Depot mappen
# ================================
# Berechnet die Distanz zu ALLEN Knoten gleichzeitig
distances_depot = vectorized_haversine(depot_lat, depot_lon, lat_series, lon_series)

# Finde den Index (Node-ID) des minimalen Wertes
depot_node_result = distances_depot.idxmin()
min_distance_depot = distances_depot.min()

print(f"Depot gemappt auf Knoten: {depot_node_result} ({round(min_distance_depot, 2)} m)")

# ================================
# 2. Kunden mappen
# ================================
customer_to_node = {}
df_customers = all_data['customer_locations'] 

for _, customer in df_customers.iterrows():
    c_lat = float(customer["Breitengrad"])
    c_lon = float(customer["Längengrad"])
    
    # Wieder für alle Knoten gleichzeitig berechnen
    distances_customer = vectorized_haversine(c_lat, c_lon, lat_series, lon_series)
    
    nearest_node = distances_customer.idxmin()
    min_dist = distances_customer.min()
    
    customer_to_node[customer["Nr"]] = {
        "customer_name": customer["Ort"],
        "nearest_node": int(nearest_node),
        "distance_m": round(min_dist, 2)
    }

# ================================
# ✅ DEBUG OUTPUT & ZIEL DEFINITION
# ================================
print("\n=== Kunden-Mapping ===")
for cid, info in customer_to_node.items():
    print(f"Kunde {cid}: {info['customer_name']} → Knoten: {info['nearest_node']} ({info['distance_m']} m)")

O_dep = {row["Nr"]: int(depot_node_result) for _, row in df_customers.iterrows()}
D_dep = {cid: info["nearest_node"] for cid, info in customer_to_node.items()}

Depot gemappt auf Knoten: 1217591 (811.47 m)

=== Kunden-Mapping ===
Kunde 1: Tankstelle Gütersloh → Knoten: 1060500 (9.97 m)
Kunde 2: Industriegebiet Chemiekonzern Hamburg → Knoten: 1858827 (741.61 m)
Kunde 3: Industriegebiet Berlin → Knoten: 89164 (634.29 m)
Kunde 4: Industriegebiet Bautzen → Knoten: 1238040 (281.51 m)
Kunde 5: Industriegebiet Bayreuth → Knoten: 1676377 (301.25 m)
Kunde 6: Industriegebiet Erding → Knoten: 974691 (689.53 m)
Kunde 7: Industriegebiet Neu-Ulm → Knoten: 1837305 (229.36 m)
Kunde 8: Industriegebiet Homburg → Knoten: 3514 (2085.23 m)
Kunde 9: Industriegebiet Hürth (Köln) → Knoten: 1130686 (587.53 m)
Kunde 10: Industriegebiet Erfurt → Knoten: 123901 (190.73 m)


In [7]:

# ALLOW DICTIONARY
# Daten laden
import logging
import sys
logger = logging.getLogger(__name__)

# 1. Tunnel-Mapping direkt aus dem Master-Graph erzeugen
# Wir nutzen zip(), da iterrows() bei 3 Millionen Kanten extrem langsam wäre!
tunnel_category_dict = {
    (u, v): tunnel_cat 
    for u, v, tunnel_cat in zip(df_master_edges['u'], df_master_edges['v'], df_master_edges['tunnel'])
}

logger.debug(f"{len(tunnel_category_dict):,} Tunnel-Mappings erstellt")

K = all_data['delivery_routes']['danger_class'].unique().tolist()

# Kanten aus dem Master-Graph laden
E = list(zip(df_master_edges['u'], df_master_edges['v']))

# OSM Tunneltypen -> ADR Kategorien

tunnel_category_mapping = {

    # Kein Tunnel
    'no': 'A',
    '': 'A',
    None: 'A',

    # Leicht restriktiv
    'building_passage': 'B',

    # Moderat restriktiv
    'covered': 'C',

    # Starke Restriktionen
    'yes': 'D',
    'avalanche_protector': 'D',
}


# ADR Restriktionen

ADR_tunnel_restrictions = {

    'A': {k: True for k in K },

    'B': {k: True for k in K},

    'C': {
        '1.1D': False,
        '1.5D': False,
        '2': True,
        '2 (TOC)': True,
        '3': True,
        '4': True,
        '5': True,
        '6': True,
        '8': True,
        '9': True,
    },

    'D': {
        '1.1D': False,
        '1.5D': False,
        '2': True,
        '2 (TOC)': True,
        '3': True,
        '4': True,
        '5': True,
        '6': False,
        '8': True,
        '9': False,
    }
}


# Allow Dictionary erzeugen

Allow_calc = {
    (e, k): 1 if ADR_tunnel_restrictions[tunnel_category_mapping.get(tunnel_category_dict.get(e, 'no'), 'A')].get(k, True) else 0
    for e in E 
    for k in K
}

logger.debug(f"Allow erstellt. Einträge: {len(Allow_calc):,}")


# Beispiele
sample = list(Allow_calc.items())[:10]

for key, value in sample:

    logger.debug(f"{key}: {value}")
    
unique_tunnel = set(df_master_edges['tunnel'].dropna().unique())
print(f"Vorhandene Tunnel-Kategorien: {unique_tunnel}")

print(unique_tunnel)

print(f"df_nodes head: \n{df_nodes.head()}")

Vorhandene Tunnel-Kategorien: {'covered', 'yes', 'building_passage', 'passage', 'no', 'culvert'}
{'covered', 'yes', 'building_passage', 'passage', 'no', 'culvert'}
df_nodes head: 
         lat        lon
0  48.226067  11.541343
1  48.226192  11.542830
2  53.470430   9.914535
3  53.470443   9.915161
4  53.470452   9.915529


In [8]:
import pulp as pl
import logging
import networkx as nx
import gc  # WICHTIG: Garbage Collector für RAM-Freigabe

logger = logging.getLogger(__name__)

if 'charging_nodes' not in globals():
    print("⚠️ Warnung: charging_nodes nicht gefunden! Setze auf leere Liste.")
    charging_nodes = set()

logger.setLevel(logging.DEBUG)
# ----- SETS (MENGEN) & PARAMETER --------#

# --- SETS (Mengen) ---
V = [row['type'] for _, row in all_data['vehicles'].iterrows()] 
logger.debug(f"Fahrzeugtypen: {V}")

L = [row['customer_id'] for _, row in all_data['delivery_routes'].iterrows()]
logger.debug(f"Lieferungen: {L}")

# Phase 2 Regel: Duplikate entfernen ist hier bereits perfekt umgesetzt!
K = list(set([row['danger_class'] for _, row in all_data['delivery_routes'].iterrows()]))
logger.debug(f"Einzigartige Gefahrgutklassen für Risiko-Berechnung: {K}")    

N = df_nodes.index.tolist()
logger.debug(f"Anzahl Knoten im Straßennetz: {len(N)}\n N Header: {N[:5]}")

E = list(zip(df_master_edges['u'], df_master_edges['v']))
logger.debug(f"Anzahl Kanten im Straßennetz: {len(E)}\n E Header: {E[:5]}")


# --- PARAMETER (Eingabedaten) ---
Dem = {row['customer_id']: row['quantity'] for _, row in all_data['delivery_routes'].iterrows()} 
Class = {row['customer_id']: row['danger_class'] for _, row in all_data['delivery_routes'].iterrows()} 

depot_node = depot_node_result
logger.debug(f"Depot-Knoten: {depot_node}")

O_dep = O_dep
logger.debug(f"Startknoten (Origin) je Lieferung: {O_dep}")

D_dep = D_dep
logger.debug(f"Zielknoten (Destination) je Lieferung: {D_dep}")

# --- CHARGE_NODE DICTIONARY ---
ChargeNode = {n: 1 if n in charging_nodes else 0 for n in N}
logger.debug(f"Anzahl der Knoten mit Ladestation im Hauptnetz: {sum(ChargeNode.values())}")

# Fahrzeugdaten
Cap = {row['type']: row['capacity_kg'] for _, row in all_data['vehicles'].iterrows()}   
Range = {row['type']: row['range_km'] for _, row in all_data['vehicles'].iterrows()}    
FC = {row['type']: row['fixed_cost'] for _, row in all_data['vehicles'].iterrows()}   

# --- Netzwerk- & Risikodaten ---
Dist = {
    (u, v): float(length) / 1000
    for u, v, length in zip(df_master_edges["u"], df_master_edges["v"], df_master_edges["length_m"])
}
logger.debug(f"Distanz-Dictionary vor Kontraktion: {len(Dist):,} Einträge")

# --- RISIKO BERECHNEN (alpha, beta, gamma) ---
alpha = 0.4
beta = 0.4
gamma = 0.2

logger.info("Berechne Risk-Dictionary für alle Kanten und Klassen über raw Numpy Arrays...")

# 🚀 PHASE 2 FIX: Reine Numpy-Mathematik ohne Pandas Overhead!
u_vals = df_master_edges['u'].values
v_vals = df_master_edges['v'].values
pop_vals = df_master_edges['population'].values
weight_vals = df_master_edges['weighted_score'].values
nature_vals = df_master_edges['nature_score'].values

temp_risk = (alpha * pop_vals) + (beta * weight_vals) + (gamma * nature_vals)

Risk = {
    ((u, v), k): r
    for u, v, r in zip(u_vals, v_vals, temp_risk)
    for k in K
}

# RAM der Vektoren sofort vernichten!
del u_vals, v_vals, pop_vals, weight_vals, nature_vals, temp_risk
gc.collect()

logger.debug(f"Risk Dictionary: {len(Risk):,} Einträge")

Allow = Allow_calc  # Wird vermutlich in einer vorherigen Zelle generiert
logger.debug(f"Allow Dictionary: {len(Allow):,} Einträge")


# -------------- Kontraktion der Kanten und Knoten -----------

G_raw = nx.DiGraph()
G_raw.add_weighted_edges_from(((u, v, d) for (u, v), d in Dist.items()))
logger.debug(f"Roher Graph: {G_raw.number_of_nodes()} Knoten, {G_raw.number_of_edges()} Kanten")

largest_cc = max(nx.strongly_connected_components(G_raw), key=len)
G_contract = G_raw.subgraph(largest_cc).copy()

# --- RAM SOFORT FREIGEBEN ---
del G_raw
del largest_cc
gc.collect()

logger.debug(f"Hauptnetzwerk/Contract: {G_contract.number_of_nodes()} Knoten, {G_contract.number_of_edges()} Kanten")

# Phase 3 Regel: charging_nodes sicher im Graph behalten
protected_nodes = set(O_dep.values()).union(set(D_dep.values())).union(charging_nodes)

# Kontraktions-Schleife
contracted_count = 0
nodes_to_check = list(G_contract.nodes())

for node in nodes_to_check:
    if node in protected_nodes:
        continue
        
    if G_contract.in_degree(node) == 1 and G_contract.out_degree(node) == 1:
        u = list(G_contract.predecessors(node))[0]
        v = list(G_contract.successors(node))[0]
        
        if u != v:
            dist_in = G_contract[u][node].get('weight', 0)
            dist_out = G_contract[node][v].get('weight', 0)
            new_dist = dist_in + dist_out
            
            new_risk = {}
            new_allow = {}
            for k in K:
                r_in = Risk.get(((u, node), k), 0)
                r_out = Risk.get(((node, v), k), 0)
                new_risk[k] = r_in + r_out
                
                a_in = Allow.get(((u, node), k), 1)
                a_out = Allow.get(((node, v), k), 1)
                new_allow[k] = a_in * a_out

            if G_contract.has_edge(u, v):
                if new_dist < G_contract[u][v]['weight']:
                    G_contract[u][v]['weight'] = new_dist
                    for k in K:
                        Risk[((u, v), k)] = new_risk[k]
                        Allow[((u, v), k)] = new_allow[k]
            else:
                G_contract.add_edge(u, v, weight=new_dist)
                for k in K:
                    Risk[((u, v), k)] = new_risk[k]
                    Allow[((u, v), k)] = new_allow[k]
            
            # 🚀 KRITISCHER RAM FIX: Alte Kanten aus Risk/Allow löschen, bevor sie ewig als Geister-Daten im RAM bleiben!
            for k in K:
                Risk.pop(((u, node), k), None)
                Risk.pop(((node, v), k), None)
                Allow.pop(((u, node), k), None)
                Allow.pop(((node, v), k), None)

            G_contract.remove_node(node)
            contracted_count += 1

logger.debug(f"Erfolgreich gelöschte Zwischenknoten: {contracted_count}")
logger.debug(f"Knoten nach Kontraktion: {len(G_contract.nodes())}")
logger.debug(f"Kanten nach Kontraktion: {len(G_contract.edges())}")


# 4. MILP-DATEN AKTUALISIEREN
E = list(G_contract.edges())
N = list(G_contract.nodes())

Dist = {
    (u, v): data['weight']
    for u, v, data in G_contract.edges(data=True)
}

node_coords = {
    idx: (float(lat), float(lon)) 
    for idx, lat, lon in zip(df_nodes.index, df_nodes["lat"], df_nodes["lon"])
}

w1 = 0.65  
w2 = 0.35  

logger.setLevel(logging.INFO)

⚠️ Warnung: charging_nodes nicht gefunden! Setze auf leere Liste.
16:28:41 - __main__ - DEBUG - Fahrzeugtypen: ['MAN_eTGX', 'Volvo_FH_Electric', 'Mercedes_eActros_600']
16:28:41 - __main__ - DEBUG - Lieferungen: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
16:28:41 - __main__ - DEBUG - Einzigartige Gefahrgutklassen für Risiko-Berechnung: ['1.1D', '2 (TOC)', '2', '9', '8', '3']
16:28:41 - __main__ - DEBUG - Anzahl Knoten im Straßennetz: 2020869
 N Header: [0, 1, 2, 3, 4]
16:28:42 - __main__ - DEBUG - Anzahl Kanten im Straßennetz: 3025675
 E Header: [(3786023396, 3786023162), (10210552, 277783046), (277783046, 10210552), (277783046, 5186817684), (5186817684, 277783046)]
16:28:42 - __main__ - DEBUG - Depot-Knoten: 1217591
16:28:42 - __main__ - DEBUG - Startknoten (Origin) je Lieferung: {1: 1217591, 2: 1217591, 3: 1217591, 4: 1217591, 5: 1217591, 6: 1217591, 7: 1217591, 8: 1217591, 9: 1217591, 10: 1217591}
16:28:42 - __main__ - DEBUG - Zielknoten (Destination) je Lieferung: {1: 1060500, 2: 1858827, 3: 

In [9]:
import pickle
import numpy as np
import networkx as nx
from scipy.spatial import KDTree
import gc

print("=== 1. ECHTE KNOTEN-IDs LADEN ===")
with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/germany_graph_with_accidents.pkl", 'rb') as f:
    temp_data = pickle.load(f)

df_nodes_orig = temp_data["nodes"]

# RAM-SCHUTZ: Das gigantische temp_data Dictionary sofort vernichten!
del temp_data
gc.collect()

# Die echte OSM-ID herausfinden
if 'osmid' in df_nodes_orig.columns:
    echte_ids = df_nodes_orig['osmid'].values
elif 'node' in df_nodes_orig.columns:
    echte_ids = df_nodes_orig['node'].values
else:
    echte_ids = df_nodes_orig.index.values

lats = df_nodes_orig['lat'].astype(float).values
lons = df_nodes_orig['lon'].astype(float).values

# Wir filtern auf Knoten, die die Kontraktion überlebt haben (N)
valid_nodes_set = set(N) 
valid_mask = [node_id in valid_nodes_set for node_id in echte_ids]

valid_ids = echte_ids[valid_mask]
valid_lats = lats[valid_mask]
valid_lons = lons[valid_mask]

print(f"Es gibt {len(valid_ids):,} intakte Knoten für das Mapping.")

print("\n=== 2. RE-MAPPING MIT ECHTEN IDs (KD-TREE) ===")
# 🔥 HIER IST DER FEHLENDE BAUM-AUFBAU:
coords = np.column_stack((valid_lats, valid_lons))
tree = KDTree(coords)

# --- DEPOT MAPPING ---
depot_lat, depot_lon = 51.3127, 9.4797
dist_depot, idx_depot = tree.query([depot_lat, depot_lon], k=1)

new_depot_node = int(valid_ids[idx_depot])
print(f"Depot gemappt auf ECHTE OSM-ID: {new_depot_node}")

O_dep = {l: new_depot_node for l in L}
D_dep = {}

# --- KUNDEN MAPPING ---
df_customers = all_data['customer_locations']
for _, customer in df_customers.iterrows():
    c_lat = float(customer["Breitengrad"])
    c_lon = float(customer["Längengrad"])
    c_id = customer["Nr"]
    
    if c_id in L:
        dist_c, idx_c = tree.query([c_lat, c_lon], k=1)
        D_dep[c_id] = int(valid_ids[idx_c])
        print(f"Kunde {c_id} gemappt auf ECHTE OSM-ID: {D_dep[c_id]}")


print("\n=== 2.5 RE-MAPPING DER LADESTATIONEN (VEKTORISIERT) ===")
charging_nodes = set()

charge_lats = df_charge['Breitengrad'].astype(float).values
charge_lons = df_charge['Längengrad'].astype(float).values
charge_coords = np.column_stack((charge_lats, charge_lons))

_, indices = tree.query(charge_coords, k=1)

for idx in indices:
    charging_nodes.add(int(valid_ids[idx]))

print(f"Es wurden {len(charging_nodes)} Ladestationen blitzschnell und erfolgreich auf das Netz gemappt.")

del coords, charge_coords, charge_lats, charge_lons
gc.collect()


print("\n=== 3. NEUER GEOGRAFIE-FILTER (DIJKSTRA-KORRIDOR) ===")
valid_le_pairs = []
edges_per_l = {l: [] for l in L}
nodes_per_l = {l: set() for l in L}
relevante_kanten = set()

for l in L:
    start, ziel = O_dep[l], D_dep[l]
    try:
        path = nx.shortest_path(G_contract, source=start, target=ziel, weight='weight')
        path_nodes = set(path)
        
        for u in path_nodes:
            for v in G_contract.successors(u):
                edges_per_l[l].append((u, v))
                valid_le_pairs.append((l, (u, v)))
                nodes_per_l[l].update([u, v])
                relevante_kanten.add((u, v))
            for v in G_contract.predecessors(u):
                edges_per_l[l].append((v, u))
                valid_le_pairs.append((l, (v, u)))
                nodes_per_l[l].update([v, u])
                relevante_kanten.add((v, u))
                
        print(f"Lieferung {l}: Pfad gefunden! ({len(path)} Knoten im Basis-Pfad)")
    except nx.NetworkXNoPath:
        print(f"Lieferung {l}: FEHLER - Kein Weg gefunden! (Kontrolliere Start/Ziel-Mappping)")

valid_ln_pairs = [(l, n) for l in L for n in nodes_per_l[l]]

# Fahrzeug-Kostenparameter laden
VC_per_km = all_data['vehicles'].set_index('type')['variable_cost_per_km'].to_dict()
Energy_per_km = all_data['vehicles'].set_index('type')['energy_kwh_per_km'].to_dict()
strompreis = 0.35

VC = {}
for v in V:
    for e in relevante_kanten: 
        distanz = Dist.get(e, 0)
        VC[(v, e)] = (distanz * VC_per_km[v]) + (distanz * Energy_per_km[v] * strompreis)

print(f"\n✅ Filter fertig! Es sind jetzt {len(relevante_kanten):,} hochrelevante Kanten übrig.")

print("\n=== 4. GEOGRAFIE-FILTER CHECK ===")
G_test = nx.DiGraph()
G_test.add_edges_from(relevante_kanten)

for l in L:
    if O_dep[l] not in G_test or D_dep[l] not in G_test:
        print(f"Lieferung {l}: FEHLER ❌ (Start/Ziel nicht in Kanten)")
    elif nx.has_path(G_test, O_dep[l], D_dep[l]):
        print(f"Lieferung {l}: GRÜNES LICHT ✅")
    else:
        print(f"Lieferung {l}: LÜCKE ❌")

=== 1. ECHTE KNOTEN-IDs LADEN ===


/tmp/ipykernel_80241/1188708136.py:9: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  temp_data = pickle.load(f)


Es gibt 1,004,216 intakte Knoten für das Mapping.

=== 2. RE-MAPPING MIT ECHTEN IDs (KD-TREE) ===
Depot gemappt auf ECHTE OSM-ID: 25765102
Kunde 1 gemappt auf ECHTE OSM-ID: 8335525118
Kunde 2 gemappt auf ECHTE OSM-ID: 1640430278
Kunde 3 gemappt auf ECHTE OSM-ID: 279829852
Kunde 4 gemappt auf ECHTE OSM-ID: 4955312564
Kunde 5 gemappt auf ECHTE OSM-ID: 256936302
Kunde 6 gemappt auf ECHTE OSM-ID: 11840731
Kunde 7 gemappt auf ECHTE OSM-ID: 11813731811
Kunde 8 gemappt auf ECHTE OSM-ID: 536454
Kunde 9 gemappt auf ECHTE OSM-ID: 2652376668
Kunde 10 gemappt auf ECHTE OSM-ID: 76996770

=== 2.5 RE-MAPPING DER LADESTATIONEN (VEKTORISIERT) ===
Es wurden 35391 Ladestationen blitzschnell und erfolgreich auf das Netz gemappt.

=== 3. NEUER GEOGRAFIE-FILTER (DIJKSTRA-KORRIDOR) ===
Lieferung 1: Pfad gefunden! (1695 Knoten im Basis-Pfad)
Lieferung 2: Pfad gefunden! (3787 Knoten im Basis-Pfad)
Lieferung 3: Pfad gefunden! (5431 Knoten im Basis-Pfad)
Lieferung 4: Pfad gefunden! (248 Knoten im Basis-Pfad)
Lie

In [10]:
logger.setLevel(logging.DEBUG)
#----------------------  MODELL INITIALISIERUNG--------------------------#
model = pl.LpProblem("Gefahrgut_Routenplanung_MILP", pl.LpMinimize)

valid_le_pairs = list(set(valid_le_pairs))
for l in L:
    edges_per_l[l] = list(set(edges_per_l[l]))

#---------------------- ENTSCHEIDUNGSVARIABLEN--------------------------#

# f[l, e]: Binär - 1, wenn Lieferung l über Kante e transportiert wird
#nutzen von valid_le_pairs aus dem Spatial Bounding, damit nur Variablen für geografisch relevante Kanten erzeugt werden
f = pl.LpVariable.dicts("Fluss", valid_le_pairs, cat='Binary')

# y[l, v]: Binär - 1, wenn Lieferung l dem Fahrzeug v zugewiesen wird
y = pl.LpVariable.dicts("Zuweisung", [(l, v) for l in L for v in V], cat='Binary')

# # z[v]: Binär - 1, wenn Fahrzeug v heute eingesetzt wird (Aktivierung)
# z = pl.LpVariable.dicts("Aktivierung", [v for v in V], cat='Binary')


# u[l, n]: Stetig - MTZ-Hilfsvariable für die Besuchsreihenfolge (Subtour-Eliminierung)
u = pl.LpVariable.dicts("MTZ", valid_ln_pairs, lowBound=0, upBound=len(N), cat='Continuous')

# --- NEU: BATTERIESTAND (State of Charge) ---
# soc[l, i]: Kontinuierlich - Akkustand von Lieferung l an Knoten i
soc = pl.LpVariable.dicts("SoC", valid_ln_pairs, lowBound=0, cat='Continuous')

# q[l, n]: Änderung hier: "Reichweite" statt "SoC" im Namen!
q = pl.LpVariable.dicts("Range", valid_ln_pairs, lowBound=0, cat='Continuous')

# c[l, n]: Binär - 1, wenn Lieferung l am Knoten n nachgeladen wird
c = pl.LpVariable.dicts("Charge", valid_ln_pairs, cat='Binary')

# CostVar[l]: Stetig - Hilfsvariable zur Linearisierung der fahrzeugspezifischen Streckenkosten
CostVar = pl.LpVariable.dicts("CostVar", L, lowBound=0, cat='Continuous')

#----------------  ZIELFUNKTION (Normiert & Gewichtet)-------------------#

#----------------  ZIELFUNKTION (Normiert & Gewichtet)-------------------#

# 1. Berechnung der theortischen Maxima (NUR für relevante Kanten!)
relevante_kanten_liste = list(relevante_kanten) # Aus dem Geografie-Filter

# Hole dir den höchsten Risikowert, der auf den gefilterten Kanten existiert
Risk_max = max(Risk.get((e, k), 0) for e in relevante_kanten_liste for k in K)
Risk_sum_max = Risk_max * len(L) * len(relevante_kanten_liste) 

# Verhindere Division durch Null, falls Risiko extrem klein ist
if Risk_sum_max == 0: Risk_sum_max = 1 

# Hole dir die maximalen variablen Kosten auf den gefilterten Kanten
Cost_max = (sum(FC[v] for v in V) 
          + sum(max(VC.get((v, e), 0) for v in V) for e in relevante_kanten_liste) * len(L))

# 2. Einzelterme aufbauen
risk_term = pl.lpSum(Risk.get((e, Class[l]), 0) * f[(l, e)] for l in L for e in edges_per_l[l])

# Fixkosten pro Fahrt anstelle von pro Fahrzeug
fixed_cost_term = pl.lpSum(FC[v] * y[(l, v)] for l in L for v in V)

# Wir nutzen ausschließlich CostVar für die variablen Kosten, wie in C10 linearisiert!
variable_cost_term = pl.lpSum(CostVar[l] for l in L)

# 3. Dem Modell hinzufügen
model += (
    (w1 / Risk_sum_max) * risk_term + 
    (w2 / Cost_max) * (fixed_cost_term + variable_cost_term)
), "Minimiere_Risiko_und_Kosten_normiert"


#--------------- NEBENBEDINGUNGEN (Constraints) --------------------------#


# --- C1: Transportpflicht ---
# Jede Lieferung muss exakt einem Fahrzeug zugewiesen werden.
for l in L:
    model += pl.lpSum(y[(l, v)] for v in V) == 1, f"C1_Transportpflicht_{l}"

# --- C2: Kapazitätsbedingung ---
# Die Summe der Liefermengen, die einem Fahrzeug zugewiesen werden, darf dessen Kapazität nicht überschreiten.
for l in L:
    for v in V:
        model += Dem[l] * y[(l, v)] <= Cap[v], f"C2_Kapazitaet_{l}_{v}"


# --- C4: Gefahrgutrestriktion ---
# Eine Kante darf von einer Lieferung nicht befahren werden, wenn sie gesperrt ist.
# (Es werden aus Performance-Gründen nur Constraints für gesperrte Kanten erzeugt).
for l in L:
    for e in edges_per_l[l]:
        if Allow.get((e, Class[l]), 1) == 0:
            model += f[(l, e)] <= 0, f"C4_Gefahrgut_Sperre_{l}_{e[0]}_{e[1]}"

# --- C5: Flusserhaltung (Network Flow Conservation) ---
# Garantiert eine zusammenhängende Route vom Start- zum Zielknoten.
for l in L:
    relevant_edges = edges_per_l[l]
    relevant_nodes = nodes_per_l[l]
    
    in_edges_l = {n: [] for n in relevant_nodes}
    out_edges_l = {n: [] for n in relevant_nodes}
    
    # HIER FIX: n1 und n2 statt u und v nutzen!
    for (n1, n2) in relevant_edges:
        in_edges_l[n2].append((n1, n2))
        out_edges_l[n1].append((n1, n2))

    flow_active = pl.lpSum(y[(l, v)] for v in V)

    for n in relevant_nodes:
        inflow = pl.lpSum(f[(l, e)] for e in in_edges_l[n])
        outflow = pl.lpSum(f[(l, e)] for e in out_edges_l[n])

        # Wir fügen die Constraint nur hinzu, wenn es noch keine für diese Kombination gibt
        c_name = f"C5_Flow_{l}_{n}"
        if n == O_dep[l]:      
            model += outflow - inflow == flow_active, c_name
        elif n == D_dep[l]:    
            model += inflow - outflow == flow_active, c_name
        else:                  
            model += inflow - outflow == 0, c_name

# --- C6: Subtour-Eliminierung (Miller-Tucker-Zemlin) ---
# Verhindert, dass der Solver isolierte, physikalisch unvollständige Kreise (Schleifen) bildet.
for l in L:
    relevant_edges = edges_per_l[l]
    M_mtz = len(nodes_per_l[l])  
    
    for (i, j) in relevant_edges:
        if j != O_dep[l]:  
            model += u[(l, i)] - u[(l, j)] + M_mtz * f[(l, (i, j))] <= M_mtz - 1, f"C6_MTZ_{l}_{i}_{j}"

# --- C7: Ladezustand (SoC Tracking) ---
for l in L:
    # Wir nehmen den MAN als Standard-Fahrzeug für die Reichweite (wie bei C2)
    max_range = float(Range[V[0]])
    M_soc = max_range + 1000 # Großes Big-M für die Logik-Schranke
    
    # 1. Der LKW verlässt das Depot immer mit vollem Akku
    model += soc[(l, O_dep[l])] == max_range, f"SoC_Start_{l}"
    
    # 2. Akkuverbrauch auf der Strecke berechnen
    for (i, j) in edges_per_l[l]:
        distanz = Dist.get((i, j), 0)
        
        # Falls Knoten j KEINE Ladestation ist: Akku nimmt exakt um die gefahrene Distanz ab
        if ChargeNode.get(j, 0) == 0:
            model += soc[(l, j)] <= soc[(l, i)] - distanz + M_soc * (1 - f[(l, (i, j))]), f"SoC_Drop_{l}_{i}_{j}"
        
        # Falls Knoten j EINE Ladestation ist: Akku darf auf maximal 'max_range' aufgeladen werden
        else:
            model += soc[(l, j)] <= max_range, f"SoC_Charge_{l}_{i}_{j}"


# --- C8: Akkukapazität ---
for l in L:
    for n in nodes_per_l[l]:
        model += q[(l, n)] <= pl.lpSum(Range[v] * y[(l, v)] for v in V), f"C8_Kapazitaet_{l}_{n}"

# --- C9: Ladeinfrastruktur ---
for l in L:
    for n in nodes_per_l[l]:
        model += c[(l, n)] <= ChargeNode.get(n, 0), f"C9_Laden_{l}_{n}"

# --- C10: Kosten-Linearisierung ---
# Großes M für Kosten, das alle Kanten abdeckt
M_cost = Cost_max 
for l in L:
    for v in V:
        route_cost = pl.lpSum(VC[(v, e)] * f[(l, e)] for e in edges_per_l[l])
        model += CostVar[l] >= route_cost - M_cost * (1 - y[(l, v)]), f"C10_CostLin_{l}_{v}"


In [11]:
# 1. Modellgröße checken (Ganz wichtig!)
anzahl_vars = len(model.variables())
anzahl_constraints = len(model.constraints)
print(f"🔥 Modell-Diagnose: {anzahl_vars:,} Variablen und {anzahl_constraints:,} Nebenbedingungen")

# 2. Solver mit Debug-Modus starten (keepFiles behält die .lp und .sol Dateien)
logger.debug("Ab jetzt startet Solver => 5 Minuten")
try:
    # keepFiles=True verhindert, dass PuLP die Beweise vernichtet!
    solver = pl.PULP_CBC_CMD(msg=1, timeLimit=300, keepFiles=True) 
    status = model.solve(solver)
    print(f"Status: {pl.LpStatus[status]}")
except Exception as e:
    print(f"🚨 Absturz: {e}")

🔥 Modell-Diagnose: 158,125 Variablen und 183,178 Nebenbedingungen
16:31:39 - __main__ - DEBUG - Ab jetzt startet Solver => 5 Minuten
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc Gefahrgut_Routenplanung_MILP-pulp.mps -sec 300 -timeMode elapsed -solve -printingOptions all -solution Gefahrgut_Routenplanung_MILP-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 183183 COLUMNS
At line 1092668 RHS
At line 1275847 BOUNDS
At line 1380741 ENDATA
Problem MODEL has 183178 rows, 158125 columns and 701239 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 300
Option for timeMode changed from cpu to elapsed
Problem is infeasible - 0.97 seconds
Option for printingOptions changed from normal to all
Total time (CPU seconds):       1.62   (Wal

In [12]:
# --------------------- MODELL LÖSEN --------------------------------#
logger.debug("Ab jetzt startet Solver => 5 Minuten")
solver = pl.PULP_CBC_CMD(msg=1, timeLimit=300)  # 5 Minuten Zeitlimit
status = model.solve(solver)
print(f"Status: {pl.LpStatus[status]}")

logger.setLevel(logging.INFO)

16:31:45 - __main__ - DEBUG - Ab jetzt startet Solver => 5 Minuten
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/e04c5a7aa5dd447fab05a56cd3408b00-pulp.mps -sec 300 -timeMode elapsed -solve -printingOptions all -solution /tmp/e04c5a7aa5dd447fab05a56cd3408b00-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 183183 COLUMNS
At line 1092668 RHS
At line 1275847 BOUNDS
At line 1380741 ENDATA
Problem MODEL has 183178 rows, 158125 columns and 701239 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 300
Option for timeMode changed from cpu to elapsed
Problem is infeasible - 1.08 seconds
Option for printingOptions changed from normal to all
Total time (CPU seconds):       1.68   (Wallclock seconds):       1.63

Status: Infeasible


In [13]:
# ----------------ERGEBNIS-AUSWERTUNG --------------------------#

if pl.LpStatus[model.status] == 'Optimal':
    logger.info("OPTIMALE LOESUNG GEFUNDEN! Werte die Routen aus...\n")

    total_vc = 0
    total_risk = 0
    summary_table_data = []

    for l in L:
        print(f"\n LIEFERUNG {l} | Gefahrgutklasse: {Class[l]} | Gewicht: {Dem[l]} kg")
        print("-" * 60)
        
        # GEÄNDERT: Float-Toleranz > 0.5 für die y-Variable
        zugewiesenes_v = None
        for v in V:
            if pl.value(y[(l, v)]) is not None and pl.value(y[(l, v)]) > 0.5:
                zugewiesenes_v = v
                break
        
        if not zugewiesenes_v:
            print("   Fehler: Kein Fahrzeug für diese Lieferung zugewiesen!")
            continue
            
        print(f"   Fahrzeug: {zugewiesenes_v} (Reichweite: {Range[zugewiesenes_v]} km, Kapazitaet: {Cap[zugewiesenes_v]} kg)")

        # 2. Welche Kanten wurden befahren? (f-Variable)
        route_edges = []
        lieferung_dist = 0
        lieferung_risk = 0
        lieferung_vc = 0
        
        for e in edges_per_l[l]:
            if pl.value(f[(l, e)]) is not None and pl.value(f[(l, e)]) > 0.5:  
                route_edges.append(e)
                lieferung_dist += Dist[e]
                lieferung_risk += Risk.get((e, Class[l]), 0)
                lieferung_vc += VC[(zugewiesenes_v, e)]
        
        # 3. Kanten in die richtige Reihenfolge bringen (Start -> Ziel)
        geordnete_route = []
        aktueller_knoten = O_dep[l]
        ziel_knoten = D_dep[l]
        
        verfuegbare_kanten = route_edges.copy()
        
        for _ in range(len(route_edges)):
            naechste_kante = next((kante for kante in verfuegbare_kanten if kante[0] == aktueller_knoten), None)
            
            if naechste_kante:
                verfuegbare_kanten.remove(naechste_kante)
                geordnete_route.append(naechste_kante)
                aktueller_knoten = naechste_kante[1] 
                
                if aktueller_knoten == ziel_knoten:
                    break
                    
        # 4. Route ausgeben
        print(f"   Route ({len(geordnete_route)} Kanten):")
        if geordnete_route:
            knoten_liste = [str(geordnete_route[0][0])] + [str(kante[1]) for kante in geordnete_route]
            chunk_size = 8
            chunks = [" -> ".join(knoten_liste[i:i+chunk_size]) for i in range(0, len(knoten_liste), chunk_size)]
            route_string = " -> \n      ".join(chunks)
            print(f"      {route_string}")
        else:
            print("      Konnte die Route nicht chronologisch sortieren (Fallback-Ansicht):")
            fallback_string = " | ".join([f"{kante[0]} -> {kante[1]}" for kante in route_edges])
            print(f"      {fallback_string}")


        # 5. Metriken pro Lieferung ausgeben
        print(f"\n   Zusammenfassung fuer Lieferung {l}:")
        print(f"      - Gesamtdistanz:      {lieferung_dist:.2f} km")
        print(f"      - Variable Kosten:    {lieferung_vc:.2f} EUR")
        print(f"      - Kumuliertes Risiko: {lieferung_risk:.4f}")
        
        total_vc += lieferung_vc
        total_risk += lieferung_risk
        
        # Daten fuer die finale Tabelle speichern
        summary_table_data.append({
            'Lieferung': l,
            'Fahrzeug': zugewiesenes_v,
            'Fixkosten': FC[zugewiesenes_v],
            'VarKosten': lieferung_vc,
            'Risiko': lieferung_risk
        })
        
    # --- Gesamtsystem-Auswertung (GEÄNDERT) ---
   
    print("\nGESAMTSYSTEM")
    
    # Welche Fahrzeugtypen wurden insgesamt (evtl. mehrfach) genutzt?
    genutzte_typen = list(set([row['Fahrzeug'] for row in summary_table_data]))
    
    # Summiere die Fixkosten aus der Tabelle (jede Fahrt kostet!)
    total_fc = sum(row['Fixkosten'] for row in summary_table_data)
    
    print(f" Eingesetzte Fahrzeugtypen: {len(genutzte_typen)}")
    print(f" Namen:                     {', '.join(genutzte_typen)}")
    print(f" Gesamte Fixkosten:         {total_fc:.2f} EUR")
    print(f" Gesamte Var. Kosten:       {total_vc:.2f} EUR")
    print(f" TOTALE KOSTEN:             {total_fc + total_vc:.2f} EUR")
    print(f" TOTALES RISIKO:            {total_risk:.4f}")
    print("="*85 + "\n")

    # --- Zusammenfassende Tabelle ---
    print(" ZUSAMMENFASSUNG: ZUWEISUNG UND KOSTEN")
    print("-" * 85)
    print(f"| {'Lieferung':<10} | {'Fahrzeug':<25} | {'Fixkosten':<10} | {'Var. Kosten':<11} | {'Risiko':<8} |")
    print("-" * 85)
    for row in summary_table_data:
        print(f"| {str(row['Lieferung']):<10} | {row['Fahrzeug']:<25} | {row['Fixkosten']:>6.2f} EUR | {row['VarKosten']:>7.2f} EUR | {row['Risiko']:>8.4f} |")
    print("-" * 85)

else:
    logger.error(f"Keine verwertbare Route gefunden. Status ist: {pl.LpStatus[model.status]}")
    if pl.LpStatus[model.status] == 'Infeasible':
        print("Status: Infeasible - Das Modell hat keine Lösung gefunden, die alle Nebenbedingungen erfüllt")

16:31:53 - __main__ - ERROR - Keine verwertbare Route gefunden. Status ist: Infeasible
Status: Infeasible - Das Modell hat keine Lösung gefunden, die alle Nebenbedingungen erfüllt


In [14]:
print("\n" + "="*80)
print("CONSTRAINT ANALYSE")
print("="*80)

for name, constraint in model.constraints.items():
  
    lhs_value = constraint.value()

    
    # Prüfe Verletzung
    violated = False

    # <=
    if constraint.sense == -1:
        # if lhs_value > 0:
        #     violated = True
        if lhs_value > 1e-4: 
            violated = True

    # >=
    elif constraint.sense == 1:
        if lhs_value < 0:
            violated = True
            

    # ==
    elif constraint.sense == 0:
        if abs(lhs_value) > 1e-6:
            violated = True

    if violated:
        print(f"\nConstraint: {name}")
        print(f"Value: {lhs_value}")
        print(f"Sense: {constraint.sense}")
        print(f"RHS: {-constraint.constant}")

        print(">>> VERLETZT <<<")


CONSTRAINT ANALYSE

Constraint: C4_Gefahrgut_Sperre_4_279391165_864404619
Value: 1.0
Sense: -1
RHS: 0
>>> VERLETZT <<<

Constraint: C5_Flow_1_2962565537
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_1_10285394639
Value: 1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_2_12386599817
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_2_1433009381
Value: 1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_3_32020816
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_3_2055657122
Value: 1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_4_3246881773
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_4_864404619
Value: 1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_5_401548647
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_5_413916634
Value: 1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_6_230073965
Value: 1.0
Sense: 0
RHS: -0.0
>>> V